# 05 — Arc-Length Continuation

**Topic**: Predictor-corrector scheme, fold tracing, adaptive step size.

**Reference**: Krack & Gross (2019) §4

**Estimated runtime**: < 30 seconds

## Theory

Arc-length (pseudo-arc-length) continuation traces the solution branch $\{(\mathbf{Q}, \Omega) : R(\mathbf{Q}, \Omega) = 0\}$ even past fold (turning) points (K&G §4).

**Predictor** (K&G §4.2): solve the bordered system for the unit tangent $(t_x, t_\lambda)$:

$$\begin{bmatrix} J & \partial R/\partial\lambda \\ t_x^\top & t_\lambda \end{bmatrix} \begin{bmatrix} t_x \\ t_\lambda \end{bmatrix} = \begin{bmatrix} 0 \\ 1 \end{bmatrix} \tag{1}$$

Then extrapolate: $x_{pred} = x + \Delta s \cdot t_x$, $\lambda_{pred} = \lambda + \Delta s \cdot t_\lambda$.

**Corrector** (K&G §4.3): Newton iteration on the augmented system:

$$G(x, \lambda) = \begin{bmatrix} R(x, \lambda) \\ (x - x_p)^\top t_x + (\lambda - \lambda_p)t_\lambda \end{bmatrix} = 0 \tag{2}$$

**Adaptive step size** (K&G §4.4): double $\Delta s$ if Newton converged in $<5$ iterations; halve if $>9$.

**Fold detection**: a sign change in $t_\lambda$ indicates a fold point (turning point on the FRF).

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

from nlvib.nonlinearities.elements import cubic_spring
from nlvib.systems.oscillators import SingleMassOscillator
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions
from nlvib.visualization.plots import plot_frf

# Duffing oscillator (K&G §5.1)
m, d, k, k3, F_amp = 1.0, 0.02, 1.0, 0.5, 0.1
H = 3
excitation = {'dof': 0, 'amplitude': F_amp, 'harmonic': 1}

system = SingleMassOscillator(m=m, d=d, k=k)
system.add_nonlinear_element(cubic_spring(k3=k3, dof_index=0))

print("System ready. Running continuation...")

In [ ]:
# Run arc-length continuation to trace the full Duffing FRF branch
# including the folded (S-curve) portion

ds_initial = 0.02  # ← try changing this
ds_max     = 0.08  # ← try changing this

omega0 = 0.6
Q0 = np.zeros(2*H + 1)
Q0[1] = F_amp / abs(-(omega0**2)*m + k + 1j*omega0*d)

def residual_fn(Q, lam):
    return hb_residual(Q, lam, system, H, excitation)

opts = ContinuationOptions(
    ds_initial=ds_initial,
    ds_min=1e-5,
    ds_max=ds_max,
    max_steps=600,
    max_newton_iter=20,
    newton_tol=1e-10,
    adapt_step=True,
    lambda_min=0.55,
    lambda_max=1.45,
)

solver = ContinuationSolver()
result = solver.run(residual_fn, Q0, omega0, opts)

print(f"Continuation: {result.n_steps} steps, converged={result.converged}")
print(f"Message: {result.message}")
print(f"Fold points (sign changes in t_lambda): {np.sum(np.diff(result.stability.astype(int)) != 0)}")

# Extract omega and fundamental amplitude
omega_arr = result.solutions[:, -1]
Q_arr     = result.solutions[:, :-1]
amp_arr   = np.sqrt(Q_arr[:, 1]**2 + Q_arr[:, 2]**2)
stab_arr  = ~result.stability  # invert: solver True = unstable

In [ ]:
# Plot FRF with predictor-corrector steps visualised
# Each accepted step is a point on the branch

@dataclass
class FRFResult:
    omega: np.ndarray
    amplitude: np.ndarray
    stability: np.ndarray

frf_result = FRFResult(omega=omega_arr, amplitude=amp_arr, stability=stab_arr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: FRF with step markers
fig2 = plot_frf(frf_result, ax=ax1)
# Overlay step markers every 5th accepted step
stride = max(1, len(omega_arr) // 20)
ax1.plot(omega_arr[::stride], amp_arr[::stride], 'k+', ms=8, zorder=5,
         label='accepted steps (every 5th)')
ax1.set_title(f'Duffing FRF — arc-length continuation\n(k3={k3}, H={H}, ds_initial={ds_initial})')
ax1.legend(fontsize=8)

# Right: adaptive step size history
ax2.plot(result.ds_history, 'o-', ms=3, color='tab:green')
ax2.axhline(ds_max, color='r', ls='--', lw=1, label=f'ds_max={ds_max}')
ax2.axhline(opts.ds_min, color='gray', ls='--', lw=1, label=f'ds_min={opts.ds_min}')
ax2.set_xlabel('Continuation step')
ax2.set_ylabel('Arc-length step $\\Delta s$')
ax2.set_title('Adaptive step size history')
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig

In [ ]:
# Show where fold points occur on the branch
stability_changes = np.where(np.diff(result.stability.astype(int)) != 0)[0]
print(f"Fold point indices: {stability_changes}")
for idx in stability_changes:
    print(f"  Fold at omega={omega_arr[idx]:.4f} rad/s, amp={amp_arr[idx]:.6f}")

# Summary
peak_idx = np.argmax(amp_arr)
print(f"\nPeak amplitude = {amp_arr[peak_idx]:.4f} at omega = {omega_arr[peak_idx]:.4f} rad/s")
print(f"Number of stable points:   {np.sum(stab_arr)}")
print(f"Number of unstable points: {np.sum(~stab_arr)}")

## Key Takeaways

- Arc-length continuation traces the **full** FRF branch including the unstable (dashed) segment between fold points.
- Adaptive step size: the solver automatically takes larger steps in smooth regions and smaller steps near folds.
- Two fold points bracket the unstable segment — the classic S-curve of the Duffing oscillator.
- Setting `ds_initial` too large can cause the solver to miss the fold; too small is slow — the adaptive scheme handles this automatically.